# CSC1110/CSC1122 Lab 2: Text Classification (Jupyter Notebook)

This notebook implements **Naïve Bayes from scratch** and **Scikit-learn's Naïve Bayes**
for the Nigerian English news classification dataset: `okite97/news-data`.

**What you'll do:**
1. Load the dataset (train/test) from Hugging Face.
2. Build a Multinomial Naïve Bayes classifier *from scratch*:
   - Compute class priors, $P(c)$.
   - Compute word likelihoods $P(w\mid c)$ with Laplace smoothing.
   - Score in **log-space** and **ignore unknown words** at test time.
3. Evaluate accuracy (and show a confusion matrix).
4. Replace with **scikit-learn** (CountVectorizer + MultinomialNB) and compare.

> Tip: Run cells top-to-bottom. If `datasets` isn't installed, the first cell will install it.

In [ ]:
# If needed, install dependencies in your environment (uncomment as required).# %pip install -q datasets pandas scikit-learn

## 1) Imports & Utility Functions

In [ ]:
import math
import re
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

TOKEN_PATTERN = re.compile(r"[A-Za-z]+")

def simple_tokenize(text: str):
    if not isinstance(text, str):
        return []
    return [t.lower() for t in TOKEN_PATTERN.findall(text)]

def concat_title_excerpt(row):
    title = row.get('title') or ''
    excerpt = row.get('excerpt') or ''
    return (title + " " + excerpt).strip()


## 2) Load Dataset (Hugging Face: `okite97/news-data`)

In [ ]:
ds = load_dataset('okite97/news-data')
list(ds.keys()), ds['train'][0]

Convert to pandas and **concatenate title + excerpt** as the document text.

In [ ]:
train_df = ds['train'].to_pandas().copy()
test_df  = ds['test'].to_pandas().copy()

# Ensure the expected columns exist: 'title', 'excerpt', 'category'
assert 'category' in train_df.columns
assert 'category' in test_df.columns

train_df['text'] = train_df.apply(concat_title_excerpt, axis=1)
test_df['text']  = test_df.apply(concat_title_excerpt, axis=1)

labels = sorted(train_df['category'].unique())
label_to_id = {c:i for i,c in enumerate(labels)}
id_to_label = {i:c for c,i in label_to_id.items()}
labels

## 3) Naïve Bayes (from scratch)

We'll implement a **Multinomial Naïve Bayes** classifier:
- Build a vocabulary from training data tokens (types).
- Compute class priors $P(c)$ from label frequencies.
- Compute $P(w\mid c)$ using **Laplace (add-α) smoothing** with α=1.0.
- Use **log-space** during scoring to avoid underflow.
- At test time, **ignore OOV tokens** (unknown words).

In [ ]:
# Tokenize train and test
train_tokens = [simple_tokenize(t) for t in train_df['text']]
test_tokens  = [simple_tokenize(t) for t in test_df['text']]

# Build vocabulary (types) from train only
vocab = set()
for toks in train_tokens:
    vocab.update(toks)
vocab = sorted(vocab)
vocab_index = {w:i for i,w in enumerate(vocab)}
V = len(vocab)
V

### 3.1) Estimate Priors $P(c)$ and Likelihoods $P(w\mid c)$

In [ ]:
alpha = 1.0  # Laplace smoothing

# Class priors
y_train = train_df['category'].tolist()
class_counts = Counter(y_train)
N = len(y_train)
priors = {c: class_counts[c] / N for c in labels}

# Word counts per class (multinomial)
word_counts_by_class = {c: Counter() for c in labels}
total_tokens_by_class = {c: 0 for c in labels}

for toks, c in zip(train_tokens, y_train):
    word_counts_by_class[c].update(toks)
    total_tokens_by_class[c] += len(toks)

# Precompute log P(w|c) with Laplace smoothing
log_likelihoods = {c: defaultdict(float) for c in labels}

for c in labels:
    total = total_tokens_by_class[c]
    for w in vocab:
        count_wc = word_counts_by_class[c][w]
        pwc = (count_wc + alpha) / (total + alpha * V)
        log_likelihoods[c][w] = math.log(pwc)

# Also store log-priors
log_priors = {c: math.log(priors[c]) for c in labels}

priors, list(log_priors.items())[:3]

### 3.2) Prediction Function

In [ ]:
def predict_doc(tokens):
    # Score each class in log-space: log P(c) + sum_{w in doc∩V} log P(w|c)
    best_c = None
    best_score = -1e100
    for c in labels:
        score = log_priors[c]
        for w in tokens:
            if w in vocab_index:  # ignore OOV
                score += log_likelihoods[c][w]
        if score > best_score:
            best_score = score
            best_c = c
    return best_c

y_pred_scratch = [predict_doc(toks) for toks in test_tokens]
y_test = test_df['category'].tolist()

acc_scratch = accuracy_score(y_test, y_pred_scratch)
acc_scratch

### 3.3) Evaluation: Accuracy, Confusion Matrix, Report

In [ ]:
print("From-scratch Naïve Bayes Accuracy:", acc_scratch)

cm = confusion_matrix(y_test, y_pred_scratch, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true:{c}" for c in labels], columns=[f"pred:{c}" for c in labels])
cm_df

In [ ]:
print(classification_report(y_test, y_pred_scratch, labels=labels, zero_division=0))

## 4) Optional: Binarized Naïve Bayes Variant
Clip within-document word counts at 1 (useful for sentiment tasks).

In [ ]:
alpha_bin = 1.0
# Rebuild counts with binarization
word_counts_by_class_bin = {c: Counter() for c in labels}
total_tokens_by_class_bin = {c: 0 for c in labels}

for toks, c in zip(train_tokens, y_train):
    unique_toks = set(toks)
    word_counts_by_class_bin[c].update(unique_toks)
    total_tokens_by_class_bin[c] += len(unique_toks)

log_likelihoods_bin = {c: defaultdict(float) for c in labels}
for c in labels:
    total = total_tokens_by_class_bin[c]
    for w in vocab:
        count_wc = word_counts_by_class_bin[c][w]
        pwc = (count_wc + alpha_bin) / (total + alpha_bin * V)
        log_likelihoods_bin[c][w] = math.log(pwc)

def predict_doc_bin(tokens):
    tokens = set(tokens)
    best_c, best_score = None, -1e100
    for c in labels:
        score = log_priors[c]
        for w in tokens:
            if w in vocab_index:
                score += log_likelihoods_bin[c][w]
        if score > best_score:
            best_score = score
            best_c = c
    return best_c

y_pred_bin = [predict_doc_bin(t) for t in test_tokens]
acc_bin = accuracy_score(y_test, y_pred_bin)
acc_bin

## 5) Scikit-learn Baseline (CountVectorizer + MultinomialNB)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

vectorizer = CountVectorizer(lowercase=True, token_pattern=r"[A-Za-z]+")
X_train = vectorizer.fit_transform(train_df['text'])
X_test  = vectorizer.transform(test_df['text'])

clf = MultinomialNB(alpha=1.0)
clf.fit(X_train, y_train)
y_pred_sklearn = clf.predict(X_test)
acc_sklearn = accuracy_score(y_test, y_pred_sklearn)
acc_sklearn

## 6) Compare Results

In [ ]:
print({
    'from_scratch_multinomial': acc_scratch,
    'from_scratch_binarized' : acc_bin,
    'scikit_learn'           : acc_sklearn,
})

---
### Notes & Extensions
- You can try different smoothing values (α).
- Consider removing extremely rare words or experimenting with stopword removal (often not necessary).
- Try using only `title`, only `excerpt`, or both concatenated (as above) and compare.
- For more detailed evaluation, compute precision/recall/F1 per class.